# Feature Engineering

This notebook implements and validates the feature transformations identified during exploratory data analysis.

The objectives are to:

- create reproducible derived features,
- replace raw variables where a more meaningful representation was identified,
- preserve structural missingness where relevant,
- avoid target-dependent transformations,
- prepare consistent feature sets for temporal cross-validation.

All transformations in this stage are derived exclusively from predictor variables.

The 2024 final holdout set remains sealed and is not loaded or inspected in this notebook.

In [31]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [32]:
import numpy as np
import pandas as pd

from src.data import load_development_data
from src.features import build_features

In [33]:
train_df = load_development_data()

TARGET = "HUMRAT_TEUNA"
YEAR = "SHNAT_TEUNA"

print("Shape:", train_df.shape)
print("Years:", sorted(train_df[YEAR].unique()))

assert set(train_df[YEAR].unique()) == {
    2020, 2021, 2022, 2023
}

Shape: (41626, 46)
Years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


## Add Engineered Features

In [34]:
features_df = build_features(train_df)

print("Before:", train_df.shape)
print("After: ", features_df.shape)

Before: (41626, 46)
After:  (41626, 63)


In [35]:
derived_features = [
    # Road context
    "ROAD_IS_URBAN",
    "ROAD_IS_INTERSECTION",
    "ROAD_CARRIAGEWAY",
    "ROAD_CARRIAGEWAY_TYPE",
    "ROAD_DEFECT",
    "ROAD_SURFACE_CONDITION",

    # Time
    "HOUR",
    "TIME_OF_DAY",
    "IS_WEEKEND",

    # Location
    "ROAD_SEGMENT",
    "GEO_GRID",

    # Environment
    "VISIBILITY_OR_LIGHTING_ISSUE",

    # Pedestrian
    "PEDESTRIAN_ACTIVITY",
    "CROSSWALK_USAGE",

    # Locality
    "LOCALITY_SIZE",
    "LOCALITY_FORM",
    "LOCALITY_SECTOR",
]

## Feature Validation

The derived features are validated before defining the final modeling feature sets.

In [36]:
assert len(features_df) == len(train_df)

In [37]:
for column in train_df.columns:
    assert features_df[column].equals(
        train_df[column]
    )

assert features_df[TARGET].equals(
    train_df[TARGET]
)

## New Features Review

### Road type

In [38]:
pd.crosstab(
    features_df["SUG_DEREH"],
    [
        features_df["ROAD_IS_URBAN"],
        features_df["ROAD_IS_INTERSECTION"],
    ],
    dropna=False,
)

ROAD_IS_URBAN        False        True        
ROAD_IS_INTERSECTION False True   False  True 
SUG_DEREH                                     
1                        0     0      0  13457
2                        0     0  14544      0
3                        0  5284      0      0
4                     8341     0      0      0

### Hour

In [39]:
features_df["HOUR"].describe()

count      41626.0
mean     13.627348
std       5.484246
min            0.0
25%           10.0
50%           14.0
75%           18.0
max           23.0
Name: HOUR, dtype: Float64

In [40]:
assert features_df["HOUR"].dropna().between(
    0, 23
).all()

In [41]:
pd.crosstab(
    features_df["HOUR"],
    features_df["TIME_OF_DAY"],
)

TIME_OF_DAY,Night / Early Morning,Morning,Afternoon,Evening,Late Evening
HOUR,,,,,
0,843,0,0,0,0
1,576,0,0,0,0
2,415,0,0,0,0
3,311,0,0,0,0
4,247,0,0,0,0
5,511,0,0,0,0
6,992,0,0,0,0
7,0,1913,0,0,0
8,0,2063,0,0,0


### Weekend

In [42]:
pd.crosstab(
    features_df["YOM_BASHAVUA"],
    features_df["IS_WEEKEND"],
    dropna=False,
)

IS_WEEKEND,False,True
YOM_BASHAVUA,,
1,6688,0
2,6225,0
3,6350,0
4,6305,0
5,6691,0
6,0,5608
7,0,3759


### Road segment

In [43]:
features_df[
    ["KVISH1", "KM", "ROAD_SEGMENT"]
].dropna().head(20)

,KVISH1,KM,ROAD_SEGMENT
0,461.0,53.0,461_0
13,444.0,356.0,444_300
17,85.0,178.0,85_100
23,5803.0,45.0,5803_0
26,3875.0,1.0,3875_0
30,6353.0,11.0,6353_0
36,4.0,1669.0,4_1600
39,5714.0,41.0,5714_0
43,402.0,21.0,402_0
45,65.0,703.0,65_700


In [44]:
features_df["ROAD_SEGMENT"].nunique(
    dropna=True
)

930

In [45]:
assert not (
    features_df["ROAD_SEGMENT"]
    .dropna()
    .str.contains("_-")
).any()

#### Road Segment Validation

One observation contains a recorded kilometer value but no road identifier, so a road segment cannot be constructed for that record.

In addition, 33 observations contain negative `KM` values. Because the meaning of negative kilometer values is not documented and standard road-segment binning would produce difficult-to-interpret negative segment identifiers, these observations are not assigned to `ROAD_SEGMENT`.

Negative kilometer values are preserved in the original data, but the derived `ROAD_SEGMENT` feature is created only when both `KVISH1` and a non-negative `KM` value are available.

### GEO_GRID

In [46]:
features_df[
    ["X", "Y", "GEO_GRID"]
].dropna().head(20)

,X,Y,GEO_GRID
0,185093.0,660450.0,180000_660000
1,192948.0,704662.0,190000_700000
2,175302.0,655930.0,170000_650000
3,190940.0,665512.0,190000_660000
4,181573.0,644722.0,180000_640000
5,178792.0,663149.0,170000_660000
6,184442.0,665775.0,180000_660000
7,197083.0,668197.0,190000_660000
8,176535.0,658856.0,170000_650000
9,193136.0,705764.0,190000_700000


In [47]:
features_df["GEO_GRID"].nunique(
    dropna=True
)

230

### Carriageway


In [48]:
pd.crosstab(
    train_df["HAD_MASLUL"],
    train_df["RAV_MASLUL"],
    dropna=False,
)

RAV_MASLUL,0,1,2,3,4,5
HAD_MASLUL,,,,,,
0,0,1012,4290,8391,298,720
1,6597,0,0,0,0,0
2,3117,0,0,0,0,0
3,12977,0,0,0,0,0
9,4224,0,0,0,0,0


In [49]:
pd.Series({
    "both_active": (
        train_df["HAD_MASLUL"].ne(0)
        & train_df["RAV_MASLUL"].ne(0)
    ).sum(),

    "both_not_applicable": (
        train_df["HAD_MASLUL"].eq(0)
        & train_df["RAV_MASLUL"].eq(0)
    ).sum(),
})

both_active            0
both_not_applicable    0
dtype: int64

### Crosswalk usage

In [50]:
assert (
    features_df.loc[
        train_df["MEKOM_HAZIYA"].eq(0),
        "CROSSWALK_USAGE",
    ]
    == "Not applicable"
).all()

In [51]:
pd.crosstab(
    train_df["MEKOM_HAZIYA"],
    features_df["CROSSWALK_USAGE"],
    dropna=False,
)

CROSSWALK_USAGE,Crosswalk,No crosswalk,Not applicable,Unknown
MEKOM_HAZIYA,,,,
0,0,0,33999,0
1,0,286,0,0
2,0,1229,0,0
3,4854,0,0,0
4,1050,0,0,0
9,0,0,0,208


### Locality Features

In [ ]:
assert (
    features_df.loc[
        train_df["ZURAT_ISHUV"].eq(99),
        [
            "LOCALITY_SIZE",
            "LOCALITY_FORM",
            "LOCALITY_SECTOR",
        ],
    ]
    .eq("Not applicable")
    .all()
    .all()
)

LOCALITY_SIZE,"10,000-19,999","100,000-199,999","2,000-4,999","20,000-49,999","200,000-499,999","5,000-9,999","50,000-99,999","500,000+",Not applicable,Other / special,Rural / special,<NA>
ZURAT_ISHUV,,,,,,,,,,,,
12.0,0,0,0,0,0,0,0,3200,0,0,0,0
13.0,0,0,0,0,11226,0,0,0,0,0,0,0
14.0,0,5082,0,0,0,0,0,0,0,0,0,0
15.0,0,0,0,0,0,0,2833,0,0,0,0,0
16.0,0,0,0,2865,0,0,0,0,0,0,0,0
17.0,267,0,0,0,0,0,0,0,0,0,0,0
18.0,0,0,0,0,0,147,0,0,0,0,0,0
19.0,0,0,115,0,0,0,0,0,0,0,0,0
25.0,0,0,0,0,0,0,288,0,0,0,0,0


LOCALITY_FORM,Not applicable,Other / special,Rural,Urban,<NA>
ZURAT_ISHUV,,,,,
12.0,0,0,0,3200,0
13.0,0,0,0,11226,0
14.0,0,0,0,5082,0
15.0,0,0,0,2833,0
16.0,0,0,0,2865,0
17.0,0,0,0,267,0
18.0,0,0,0,147,0
19.0,0,0,0,115,0
25.0,0,0,0,288,0


LOCALITY_SECTOR,Jewish,Non-Jewish,Not applicable,Other / unspecified,<NA>
ZURAT_ISHUV,,,,,
12.0,3200,0,0,0,0
13.0,11226,0,0,0,0
14.0,5082,0,0,0,0
15.0,2833,0,0,0,0
16.0,2865,0,0,0,0
17.0,267,0,0,0,0
18.0,147,0,0,0,0
19.0,115,0,0,0,0
25.0,0,288,0,0,0


In [54]:
assert (
    features_df.loc[
        train_df["ZURAT_ISHUV"].eq(99),
        [
            "LOCALITY_SIZE",
            "LOCALITY_FORM",
            "LOCALITY_SECTOR",
        ],
    ]
    .eq("Not applicable")
    .all()
    .all()
)

### PEDESTRIAN ACTIVITY

In [56]:
pd.crosstab(
    train_df["LO_HAZA"],
    features_df["PEDESTRIAN_ACTIVITY"],
    dropna=False,
)

PEDESTRIAN_ACTIVITY,Other,Playing,Roadside / median,Standing,Unknown,Walking
LO_HAZA,,,,,,
1,0,0,0,0,0,40
2,0,0,0,0,0,12
3,0,9,0,0,0,0
4,0,0,0,152,0,0
5,0,0,33,0,0,0
6,0,0,364,0,0,0
7,365,0,0,0,0,0
9,0,0,0,0,40651,0


## Missingness

In [57]:
(
    features_df[derived_features]
    .isna()
    .sum()
    .to_frame("missing")
)

,missing
ROAD_IS_URBAN,0
ROAD_IS_INTERSECTION,0
ROAD_CARRIAGEWAY,0
ROAD_CARRIAGEWAY_TYPE,0
ROAD_DEFECT,0
ROAD_SURFACE_CONDITION,0
HOUR,0
TIME_OF_DAY,0
IS_WEEKEND,0
ROAD_SEGMENT,28133


## Cardinality

In [58]:
pd.DataFrame({
    "n_unique": [
        features_df[feature].nunique(dropna=True)
        for feature in derived_features
    ],
    "missing": [
        features_df[feature].isna().sum()
        for feature in derived_features
    ],
}, index=derived_features)

,n_unique,missing
ROAD_IS_URBAN,2,0
ROAD_IS_INTERSECTION,2,0
ROAD_CARRIAGEWAY,3,0
ROAD_CARRIAGEWAY_TYPE,9,0
ROAD_DEFECT,3,0
ROAD_SURFACE_CONDITION,4,0
HOUR,24,0
TIME_OF_DAY,5,0
IS_WEEKEND,2,0
ROAD_SEGMENT,930,28133


## Distribution

In [59]:
for feature in derived_features:
    print(f"\n{feature}")

    display(
        features_df[feature]
        .value_counts(
            dropna=False
        )
        .to_frame("count")
    )


ROAD_IS_URBAN


,count
ROAD_IS_URBAN,
True,28001
False,13625



ROAD_IS_INTERSECTION


,count
ROAD_IS_INTERSECTION,
False,22885
True,18741



ROAD_CARRIAGEWAY


...

,count
LOCALITY_SIZE,
Not applicable,13625
"200,000-499,999",11226
"100,000-199,999",5082
"20,000-49,999",3674
"500,000+",3200
"50,000-99,999",3121
"10,000-19,999",720
Rural / special,458
"5,000-9,999",274



LOCALITY_FORM


,count
LOCALITY_FORM,
Urban,27474
Not applicable,13625
Rural,458
Other / special,65
<NA>,4



LOCALITY_SECTOR


,count
LOCALITY_SECTOR,
Jewish,26155
Not applicable,13625
Non-Jewish,1770
Other / unspecified,72
<NA>,4


## Feature Engineering Summary

The feature-engineering stage converts selected raw variables into more compact, interpretable, and model-friendly representations while preserving the original columns for later feature-set comparison.

The transformations are deterministic and target-independent, allowing the same logic to be applied consistently within temporal cross-validation and, later, to the sealed 2024 holdout set.

No target statistics or future-year information are used when constructing the derived features.

In [60]:
feature_summary = pd.DataFrame({
    "feature": derived_features,
    "dtype": [
        str(features_df[feature].dtype)
        for feature in derived_features
    ],
    "unique_values": [
        features_df[feature].nunique(dropna=True)
        for feature in derived_features
    ],
    "missing_values": [
        features_df[feature].isna().sum()
        for feature in derived_features
    ],
})

feature_summary

,feature,dtype,unique_values,missing_values
0,ROAD_IS_URBAN,boolean,2,0
1,ROAD_IS_INTERSECTION,boolean,2,0
2,ROAD_CARRIAGEWAY,string,3,0
3,ROAD_CARRIAGEWAY_TYPE,string,9,0
4,ROAD_DEFECT,string,3,0
5,ROAD_SURFACE_CONDITION,string,4,0
6,HOUR,Int64,24,0
7,TIME_OF_DAY,category,5,0
8,IS_WEEKEND,boolean,2,0
9,ROAD_SEGMENT,string,930,28133
